In [ ]:
# ── CELL 1: Verifikasi Environment ───────────────
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print(os.environ.get("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "NOT SET"))

import tfx
print("TFX version:", tfx.__version__)
import tensorflow as tf
print("TF version:", tf.__version__)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ── CELL 2: Import Library ───────────────────────
import os
import json
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tfx

from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen,
    ExampleValidator, Transform, Trainer,
    Evaluator, Pusher, Tuner
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)

print("Semua library berhasil diimport!")

In [ ]:
# ── CELL 3: Setup Path ───────────────────────────
PROJECT_ROOT = os.getcwd()
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
TRANSFORM_MODULE = os.path.join(PROJECT_ROOT, "modules", "rahadianivan09_transform.py")
TRAINER_MODULE = os.path.join(PROJECT_ROOT, "modules", "rahadianivan09_trainer.py")
PIPELINE_ROOT = os.path.join(PROJECT_ROOT, "rahadianivan09-pipeline")
METADATA_PATH = os.path.join(PIPELINE_ROOT, "metadata", "metadata.db")
SERVING_MODEL = os.path.join(PIPELINE_ROOT, "serving_model")

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(PIPELINE_ROOT, exist_ok=True)
os.makedirs(os.path.dirname(METADATA_PATH), exist_ok=True)
os.makedirs(SERVING_MODEL, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data         : {DATA_ROOT}")
print(f"Pipeline root: {PIPELINE_ROOT}")
print(f"Serving model: {SERVING_MODEL}")

In [ ]:
# ── CELL 4: Load Dataset ─────────────────────────
df = pd.read_csv(os.path.join(DATA_ROOT, "bank.csv"))

# Konversi hanya kalau masih berupa string yes/n
if df['deposit'].dtype == object:
    df['deposit'] = (df['deposit'] == 'yes').astype(int)
    df.to_csv(os.path.join(DATA_ROOT, "bank.csv"), index=False)

print(f"Total data      : {len(df)}")
print(f"Deposit (1=yes) : {df['deposit'].sum()}")
print(f"No deposit (0)  : {len(df) - df['deposit'].sum()}")
df.head()

In [ ]:
# ── CELL 5: InteractiveContext ───────────────────
context = InteractiveContext(
    pipeline_name="bank-deposit",
    pipeline_root=PIPELINE_ROOT,
    metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(
        METADATA_PATH
    ),
)
print("InteractiveContext berhasil dibuat!")

In [ ]:
# ── CELL 6: ExampleGen ───────────────────────────
example_gen = CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)
print("ExampleGen selesai!")

In [ ]:
# ── CELL 7: StatisticsGen ────────────────────────
statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)
context.run(statistics_gen)
context.show(statistics_gen.outputs["statistics"])

In [ ]:
# ── CELL 8: SchemaGen ────────────────────────────
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"]
)
context.run(schema_gen)
context.show(schema_gen.outputs["schema"])

In [ ]:
# ── CELL 9: ExampleValidator ────────────────────
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)
context.run(example_validator)
context.show(example_validator.outputs["anomalies"])

In [ ]:
# ── CELL 10: Transform ───────────────────────────
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=os.path.abspath(TRANSFORM_MODULE),
)
context.run(transform)
print("Transform selesai!")

In [ ]:
# ── CELL 11: Tuner ───────────────────────────────
tuner = Tuner(
    module_file=os.path.abspath(TRAINER_MODULE),
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=200),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=100),
)
context.run(tuner)
print("Tuner selesai!")

In [ ]:
# ── CELL 12: Trainer ─────────────────────────────
trainer = Trainer(
    module_file=os.path.abspath(TRAINER_MODULE),
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    hyperparameters=tuner.outputs["best_hyperparameters"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=100),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=50),
)
context.run(trainer)
print("Trainer selesai!")

In [ ]:
# ── CELL 13: Resolver ────────────────────────────
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
)
context.run(model_resolver)
print("Resolver selesai!")

In [ ]:
# ── CELL 14: Evaluator ───────────────────
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            label_key="deposit",
            signature_name="serving_default",
        )
    ],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(
                    class_name="BinaryAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.75}  # harus >= 75% accuracy
                        )
                    )
                ),
                tfma.MetricConfig(
                    class_name="AUC",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.70}  # harus >= 0.70 AUC
                        )
                    )
                ),
                tfma.MetricConfig(class_name="Precision"),
                tfma.MetricConfig(class_name="Recall"),
                tfma.MetricConfig(class_name="ExampleCount"),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    eval_config=eval_config,
)
context.run(evaluator)
context.show(evaluator.outputs["evaluation"])
print("Evaluator selesai!")

In [ ]:
# ── CELL 15: Pusher ──────────────────────────────
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL
        )
    ),
)
context.run(pusher)
print(f"Pusher selesai! Model tersimpan di: {SERVING_MODEL}")

In [ ]:
# ── CELL 16: Cek Hasil Pusher ────────────────────
for root, dirs, files in os.walk(SERVING_MODEL):
    level = root.replace(SERVING_MODEL, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")